# Seção 8 (validadores customizados) — pegadinhas, efeitos colaterais e caveats

Complemento da seção: o quadro abaixo reúne as **notas de revisão** já registradas nos notebooks desta pasta, e em seguida vêm
pegadinhas **novas**, cada uma com demonstração executável e a forma de evitar. Teoria da seção: [`00_teoria.md`](00_teoria.md).

## Já registradas nesta seção

| notebook | registro |
|---|---|
| [`08_projeto_solucao_instrutor`](08_projeto_solucao_instrutor.ipynb) | `data["manufactured_date"] > value` com `value=None` (por `validate_default`) → `TypeError` cru |
| [`08_projeto_solucao_instrutor`](08_projeto_solucao_instrutor.ipynb) | `id_` continua `UUID4 \| None` |
| [`09_projeto_solucao_comentada`](09_projeto_solucao_comentada.ipynb) | `model_validator(mode="after")` em atribuição deixa a instância inconsistente |

## Novas

**1. `assert` em validador some com `python -O`.**

In [1]:
import subprocess
import sys

validator_code = """
from pydantic import BaseModel, field_validator

class Signup(BaseModel):
    age: int

    @field_validator("age")
    @classmethod
    def adult(cls, value):
        assert value >= 18, "menor de idade"
        return value

try:
    print("aceito:", Signup(age=10))
except Exception as ex:
    print("recusado:", type(ex).__name__)
"""
for flags in ([], ["-O"]):
    result = subprocess.run([sys.executable, *flags, "-c", validator_code], capture_output=True, text=True)
    print(f"python {' '.join(flags) or '(padrão)'}: {result.stdout.strip()}")

python (padrão): recusado: ValidationError
python -O: aceito: age=10


O Pydantic converte `AssertionError` em erro de validação — mas com `-O` (ou `PYTHONOPTIMIZE`), o Python **remove** os `assert`.
A regra desaparece em produção sem nenhum aviso. Em validadores, sempre `raise ValueError(...)`.

**2. `info.data[...]` escapa como `KeyError` quando o campo anterior falhou.**

In [2]:
from pydantic import BaseModel, ValidationError, ValidationInfo, field_validator


class Window(BaseModel):
    start: int
    end: int

    @field_validator("end")
    @classmethod
    def after_start(cls, value: int, info: ValidationInfo) -> int:
        if value < info.data["start"]:
            raise ValueError("end antes de start")
        return value


try:
    Window(start="não é número", end=5)
except Exception as ex:
    print(f"{type(ex).__name__}: {ex}")

KeyError: 'start'


O usuário esperava "start: Input should be a valid integer" e recebe um `KeyError` (500 numa API). Use `info.data.get("start")`
e pule a regra quando o campo de origem não estiver lá — o erro dele já foi reportado.

**3. Validador *before* que muta a entrada altera os dados de quem chamou.**

In [3]:
class Ticket(BaseModel):
    tags: list[str]

    @field_validator("tags", mode="before")
    @classmethod
    def sort_tags(cls, value):
        if isinstance(value, list):
            value.sort()                                # muta o objeto recebido
        return value


caller_tags = ["urgente", "cobrança", "api"]
Ticket(tags=caller_tags)
print(f"caller_tags depois da validação: {caller_tags}")

caller_tags depois da validação: ['api', 'cobrança', 'urgente']


O validador *before* recebe o **mesmo** objeto que o chamador passou. Devolva um objeto novo (`return sorted(value)`) em vez de
modificar o recebido.

## Veja também

- [`pydantic/02_pydantic_intermediario`](../../02_pydantic_intermediario.ipynb) (caveats): validador de campo não roda no default;
  `model_validator` + `validate_assignment` grava antes de reclamar.